# cli

> Ramabana in a terminal: a transcript of blocks, a status bar, and one line to type in.

Built on [teleprint](https://github.com/answerdotai/teleprint), whose centre is the same as
this app's: an append-mostly transcript whose durable rendering is the terminal's own
scrollback. Tool calls are *blocks* rather than lines, which is what makes them foldable --
the answer stays readable and the thirty tool results it took are one click away.

In [ ]:
#| default_exp cli

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
import asyncio, tempfile, threading
from fastcore.test import test_eq, test_fail
from teleprint.keys import Key
from teleprint.testing import EmuTty
from ramabana.testing import FullHost, fake_agent

In [ ]:
#| export
import asyncio, shlex, sys, threading
from dataclasses import dataclass
from pathlib import Path
from rich.text import Text
from rich.markdown import Markdown
from rich.cells import cell_len
from rich.theme import Theme
from fastcore.script import call_parse
from fastcore.basics import patch
from teleprint.buffer import Buffer
from teleprint.compositor import Compositor
from teleprint.transcript import TranscriptView
from teleprint.tty import RealTty
from ramabana.core import agent_err
from ramabana.tools import WRITE_TOOLS, LocalHost
from ramabana.agent import Agent, Approvals, answer_md

## Blocks and keys

Six kinds of block, one gutter each. Tool results remain separate blocks so they can fold
without burying the reply that follows them.

In [ ]:
#| export
KAKU = {
    'bg0': '#111210', 'bg1': '#191b18', 'bg2': '#30332d',
    'fg0': '#f2f0e8', 'fg1': '#d6d3c8', 'gray': '#7c8077',
    'red': '#c97970', 'green': '#91aa8f', 'yellow': '#c3a66f',
    'blue': '#9bb59a', 'aqua': '#b8c7b5', 'orange': '#b49a7b',
}
GRUVBOX = KAKU  # compatibility name for extensions; the active palette is Kaku

MARKDOWN_THEME = Theme({
    'markdown.h1': f"bold {GRUVBOX['blue']}",
    'markdown.h2': f"bold {GRUVBOX['yellow']}",
    'markdown.h3': f"bold {GRUVBOX['aqua']}",
    'markdown.h4': f"bold {GRUVBOX['blue']}",
    'markdown.h5': f"bold {GRUVBOX['blue']}",
    'markdown.h6': GRUVBOX['gray'],
    'markdown.code': GRUVBOX['aqua'],
    'markdown.link': f"underline {GRUVBOX['blue']}",
    'markdown.link_url': GRUVBOX['gray'],
    'markdown.block_quote': GRUVBOX['gray'],
    'markdown.item.bullet': f"bold {GRUVBOX['blue']}",
    'markdown.item.number': f"bold {GRUVBOX['blue']}",
    'markdown.hr': GRUVBOX['bg2'],
})

GUTTERS = {
    'user':  (Text('▌ ', style=f"bold {GRUVBOX['blue']}"), Text('  ')),
    'reply': (Text('  '),                                      Text('  ')),
    'tool':  (Text('│ ', style=GRUVBOX['blue']),               Text('  ')),
    'ask':   (Text('◆ ', style=f"bold {GRUVBOX['yellow']}"), Text('  ')),
    'note':  (Text('· ', style=GRUVBOX['gray']),               Text('  ')),
    'error': (Text('× ', style=f"bold {GRUVBOX['red']}"),  Text('  ')),
}

FOLD = 12          # a block taller than this is born folded; ctrl-O opens it
HELP = """normal  enter send · ↑/↓ prompt history · ctrl+r enter transcript · ctrl+o fold · ctrl+c stop · ctrl+d quit
transcript (after ctrl+r)  ↑/↓ blocks · pgup/pgdn page · /? search · n/N matches · g/G ends · y copy · i compose · esc leave
edit    ctrl+a/e ends · ctrl+u/k cut line · ctrl+w cut word · ctrl+y paste · paste image path to attach
approve y approve · n refuse · a approve all · or type a reason and press enter to refuse
options ↑/↓ move · enter choose · an option's own letter picks it · esc cancel and keep the line
extra   /models · /model NAME · /sessions · /resume [ID|latest] · /cost · /compact · /reload"""


In [ ]:
print(HELP)

keys    enter send · ctrl+o fold the last block · ctrl+c stop the turn · ctrl+d quit
approve y approve · n refuse · a approve all · or type a reason and press enter to refuse
extra   /help, and every ramabana command: /model /cost /compact /skills /tools /reload


## The turn

`agent.stream` is a blocking generator on the model's thread, so its chunks come back over a
queue rather than being awaited. The loop has to stay free the entire time: an approval that
cannot be answered until the turn ends is not an approval, and a tool call that cannot
repaint until then is not a live feed.

In [ ]:
#| export
@dataclass
class Option:
    "One choice on the options row: the key that picks it, what it says, and what it asks for."
    key: str                # the letter that picks it directly
    label: str              # one word, in the column
    note: str = ''          # what choosing it means, in a few more
    suffix: str = ''        # appended to the prompt; `None` drops the prompt instead


#: Refactoring is where "how" is a real decision and the wrong answer is expensive: a
#: refactor the user wanted planned but got applied is a diff they now have to read, and one
#: they wanted applied but got planned is a turn they have to spend again. The suffixes are
#: instructions rather than hints, because a preference the model may weigh against the rest
#: of its briefing is not a choice the user made.
REFACTOR = (
    Option('a', 'apply', 'edit the files, then run the project’s checks',
           '\n\nMake the change now. Read each file before you edit it, keep every edit narrow, then '
           'run the project’s own tests or linter with run_shell and report exactly what passed.'),
    Option('p', 'plan', 'read only, propose the steps',
           '\n\nDo not edit anything. Read the relevant code and reply with a numbered plan: the files '
           'and symbols involved, the change to each, its risk, and the order to do them in.'),
    Option('s', 'scope', 'name what is in and out, then stop',
           '\n\nAnswer only this, with no edits and no plan: which files and symbols are in scope for '
           'this change, and which nearby ones are deliberately not. One line each.'),
    Option('c', 'cancel', 'put the line back to edit', None),
)

#: Which prompts earn a row, and which row. Deliberately a short list: a menu in front of an
#: ordinary question is a keystroke tax rather than a feature.
MENUS = ((('refactor', 'restructure', 'reorganise', 'reorganize', 'rewrite this',
           'clean up this', 'tidy up this'), 'how should I take this?', REFACTOR),)


def options_for(text):
    "`(title, options)` for a prompt worth asking *how* about, or None for one that is not."
    p = str(text or '').lower()
    return next(((title, opts) for words, title, opts in MENUS if any(w in p for w in words)), None)


class ChoiceMenu:
    """The options row: one line per `Option`, above the input line.

    It owns navigation and selection only. What a chosen option *means* is `Ui`'s business,
    which is what keeps this reusable for the next question that has more than one honest
    answer.
    """

    def __init__(self, title, options):
        self.title, self.options, self.at = title, list(options), 0

    def render(self):
        rows = [Text(f' {self.title}', style=f"bold {GRUVBOX['fg0']}")]
        width = max(len(o.label) for o in self.options)
        for i, o in enumerate(self.options):
            on = i == self.at
            row = Text(' › ' if on else '   ', style=f"bold {GRUVBOX['yellow']}")
            row.append(f'{o.key}  ', style=f"bold {GRUVBOX['yellow'] if on else GRUVBOX['gray']}")
            row.append(o.label.ljust(width), style=GRUVBOX['fg0'] if on else GRUVBOX['fg1'])
            if o.note: row.append(f'   {o.note}', style=GRUVBOX['gray'])
            rows.append(row)
        return rows

    def choose(self, key):
        "`(done, option)` for one keystroke. A `None` option means the prompt was dropped."
        if key in ('escape', 'ctrl+c'): return True, None
        if key in ('up', 'left'): self.at = (self.at - 1) % len(self.options)
        elif key in ('down', 'right'): self.at = (self.at + 1) % len(self.options)
        elif key == 'enter': return True, self.options[self.at]
        elif key.isdigit() and 0 < int(key) <= len(self.options): return True, self.options[int(key) - 1]
        elif (hit := next((o for o in self.options if o.key == key.lower()), None)) is not None:
            return True, hit
        return False, None


async def run_turn(ui, prompt):
    """One turn, streamed into the transcript.

    The agent's `stream` is a blocking generator on the model's own thread, so the chunks
    come back over a queue rather than being awaited: the loop has to stay free the whole
    time, or an approval could never be answered and a tool call could never repaint.
    """
    loop, q = asyncio.get_running_loop(), asyncio.Queue()
    def pump():
        try:
            for chunk in ui.agent.stream_with(prompt, image=ui.image):
                loop.call_soon_threadsafe(q.put_nowait, chunk)
        except Exception as e: loop.call_soon_threadsafe(q.put_nowait, agent_err(e))
        finally:
            ui.image = None
            loop.call_soon_threadsafe(q.put_nowait, None)
    threading.Thread(target=pump, daemon=True).start()
    blk = None
    try:
        while (chunk := await q.get()) is not None: blk = ui.stream(blk, chunk)
    finally:
        ui.turn = None
        for p in ui.agent.problems: ui.say(Text(p), 'error')
        ui.agent.clear_problems()
        ui.paint()
    return blk

## The surface

`Ui` is the whole terminal surface, and every method on it is synchronous and free of tty
work -- which is what lets the tests below drive it against an emulated terminal instead of
mocking one. The one hazard it has to handle is thread affinity: activity and approval
callbacks arrive on the model's worker thread, and a compositor may only be touched from the
loop thread, so everything they do goes through `_post`.

It carries three responsibilities: painting the tail (status bar plus input line), turning
tool calls and approval requests into blocks, and deciding what one keystroke means -- which
is either a coroutine for the caller to spawn, `'quit'`, or nothing.

In [ ]:
#| export
class Ui:
    """The terminal surface: a transcript of blocks, a status bar, and one line to type in.

    Every method here is synchronous and free of tty work, so the whole surface can be
    driven in a test against an emulated terminal -- which is why the async loop below is
    as small as it is.

    Callbacks arrive from the model's worker thread (`Activity.on_change`, `Approvals`), and
    a compositor may only be touched from the loop thread, so everything they do goes
    through `_post`. Without a loop registered it calls straight through, which is what
    makes the synchronous tests possible.
    """

    def __init__(self, comp, agent, loop=None):
        self.comp, self.agent, self.loop = comp, agent, loop
        comp.console.push_theme(MARKDOWN_THEME)
        self.buf = Buffer()
        self.ask = None            # the `Ask` waiting on an answer, or None
        self.turn = None           # the running turn's task, or None
        self.acts = {}             # act id -> its block
        self.hint = ''
        self.image = None           # (name, bytes) attached by a pasted path, or None
        self.frame = 0             # animated status frame; advanced only while a turn runs
        self._reply = ''           # the reply text so far, so a streamed block repaints whole
        self.history, self.history_at, self.draft = [], 0, ''
        self.menu = None            # the open `ChoiceMenu`, or None
        self.menu_prompt = ''       # the line it is asking about
        self.transcript = TranscriptView(comp, self.tail)
        agent.activity.on_change = self.on_act
        if agent.approvals is not None: agent.approvals.listen(self.on_ask, self.on_answer)

    def _post(self, fn, *a):
        "Run `fn` on the loop thread, or now when there is no loop (a test, or startup)."
        if self.loop is None: return fn(*a)
        self.loop.call_soon_threadsafe(fn, *a)

    # -- the tail ------------------------------------------------------------
    SPINNER = '⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏'

    def status(self):
        "The status bar: what is loaded, whether it is working, and what it has cost."
        s = self.agent.status()
        busy = self.turn is not None or s['busy']
        state = 'working' if busy else ('ready' if s['ready'] else 'idle')
        color = GRUVBOX['yellow'] if busy else GRUVBOX['green'] if s['ready'] else GRUVBOX['gray']
        mark = self.SPINNER[self.frame % len(self.SPINNER)] if busy else '●'
        out = Text('RAMABANA', style=f"bold {GRUVBOX['fg0']}")
        out.append(f"  {s['model']}  ", style=GRUVBOX['gray'])
        out.append(f"{mark} {state.lower()}", style=color)
        bits = [f"{s['ntools']} tools", f"{s['nskills']} skills", f"{round(s['pct_full'] * 100)}% ctx"]
        if s['compactions']: bits.append(f"{s['compactions']} compacted")
        if self.agent.use.total: bits.append(s['usage'])
        if s['problems']: bits.append(f"{len(s['problems'])} problems")
        out.append('  ' + ' · '.join(bits), style=GRUVBOX['gray'])
        return out

    async def animate(self):
        "Repaint the live tail while a turn is running; the transcript remains untouched."
        while True:
            await asyncio.sleep(0.1)
            if self.turn is not None:
                self.frame += 1
                self.paint()

    #: The approval prompt. `y`/`n`/`a` answer only while nothing is typed, because a reason
    #: like "put it in docs/ instead" contains all three letters -- so once there is text, the
    #: letters are text and `enter` is the answer.
    ASKING = 'approve? [y/n/a, or a reason + enter] '

    def prompt(self):
        "The input line: an approval question when one is pending, otherwise the prompt."
        if self.ask is not None:
            return Text(self.ASKING, style=f"bold {GRUVBOX['yellow']}") + Text(self.buf.text, style=GRUVBOX['fg0'])
        return Text('▌ ', style=f"bold {GRUVBOX['blue']}") + Text(self.buf.text, style=GRUVBOX['fg0'])

    def overlay(self):
        "The transient rows above the tail: the options row when one is open, nothing when not."
        return self.menu.render() if self.menu is not None else []

    def paint(self):
        rows, cursor = self.tail()
        self.comp.set_tail(*rows, cursor=cursor, over=self.overlay())

    # -- the transcript ------------------------------------------------------
    def say(self, body, kind='reply', fold=FOLD):
        "Print one block. Strings stay literal; explicit Rich renderables keep their styling."
        body = Text(body) if isinstance(body, str) else body
        return self.comp.print_block(body, gutter=GUTTERS.get(kind, GUTTERS['reply']),
                                     tag=kind, collapse_at=fold)

    def on_act(self, act):
        "Called twice per tool call, from the model's thread: once running, once finished."
        self._post(self._act, act)

    def _act(self, act):
        color = GRUVBOX['blue'] if act.ok is None else GRUVBOX['green'] if act.ok else GRUVBOX['red']
        line = Text(act.line(), style=color)
        blk = self.acts.get(act.id)
        if blk is None:
            self.acts[act.id] = self.say(line, 'tool')
            return self.paint()
        body = [line] if not act.detail else [line, Text(act.detail, style=GRUVBOX['gray'])]
        self.comp.set_body(blk, *body)
        self.comp.refresh_block(blk)
        self.paint()

    # -- approvals -----------------------------------------------------------
    def on_ask(self, ask):
        "A write is waiting on a person. Print what it would do, and take over the input line."
        self._post(self._ask, ask)

    def _ask(self, ask):
        self.ask = ask
        self.buf.clear()
        title = Text(ask.summary, style=f"bold {GRUVBOX['yellow']}")
        self.say(title + Text('\n\n') + Text(ask.preview, style=GRUVBOX['fg1']), 'ask', fold=None)
        self.paint()

    def on_answer(self, ask):
        self._post(self._answered, ask)

    def _answered(self, ask):
        if self.ask is not None and self.ask.id == ask.id: self.ask = None
        self.say(Text(answer_md(ask).replace('**', '')), 'note')
        self.paint()

    def answer(self, ok, session=False):
        """Answer the pending request, using whatever has been typed as the reason.

        A refusal with a reason is the point of the gate: it reaches the model, which can
        change approach instead of retrying the same edit.
        """
        if self.ask is None: return None
        note, self.buf.text = self.buf.text.strip(), ''
        return self.agent.approvals.answer(self.ask.id, ok, note, session=session)

    # -- input ---------------------------------------------------------------
    def submit(self):
        """Handle the typed line. Returns a coroutine for a turn, or None when it was handled here.

        Slash commands are answered by the agent, so every command the IDE has works here
        too -- there is one implementation of `/model`, and it is not in a frontend. They are
        recognised before the options row, or a `/model` with the word "refactor" in it would
        open a menu instead of running.
        """
        line = self.buf.text.strip()
        self.buf.clear()
        if not line: return None
        if not line.startswith('/') and (opts := options_for(line)) is not None:
            self.menu_prompt, self.menu = line, ChoiceMenu(*opts)
            return None
        self.say(Text(line), 'user')
        if line in ('/help', '/?'):
            self.say(Text(HELP), 'note', fold=None)
            return None
        if line.startswith('/'):
            out = self.agent.command(line)
            self.say(Text(out) if out is not None else Text(f'unknown command: {line}'),
                     'note' if out is not None else 'error', fold=None)
            return None
        return run_turn(self, line)

    def on_key(self, k):
        "One keystroke. Returns a coroutine to spawn, `'quit'`, or None."
        if self.menu is not None:
            done, choice = self.menu.choose(k.name)
            if not done: return self.paint()
            prompt, self.menu_prompt, self.menu = self.menu_prompt, '', None
            if choice is None or choice.suffix is None:
                # Cancelling hands the line back rather than throwing it away: the user typed
                # it, and the menu was the harness's question, not theirs.
                self.buf.text, self.buf.cursor = prompt, len(prompt)
                return self.paint()
            self.say(Text(prompt), 'user')
            self.say(Text(f'{choice.label} — {choice.note}'), 'note')
            self.paint()
            return run_turn(self, prompt + choice.suffix)
        if self.ask is not None:
            bare = not self.buf.text.strip()
            if bare and k.name in ('y', 'Y'):     self.answer(True)
            elif bare and k.name in ('n', 'N'):   self.answer(False)
            elif bare and k.name in ('a', 'A'):   self.answer(True, session=True)
            elif k.name == 'enter':               self.answer(False)   # a typed reason is a refusal
            elif k.name == 'ctrl+y':              self.answer(True)    # ...unless approved with it as guidance
            elif k.name == 'ctrl+c':              self.answer(False)     # stopping the turn refuses what it was waiting on
            else: self.buf.handle(k)
            return self.paint()
        if k.name == 'ctrl+d' and not self.buf.text: return 'quit'
        if k.name == 'ctrl+c':
            if self.turn is not None:
                self.agent.cancel()
                self.turn.cancel()
                self.turn = None
                self.say(Text('stopped'), 'note')
            self.buf.clear()
            return self.paint()
        if k.name == 'enter':
            coro = self.submit()
            self.paint()
            return coro
        if k.name == 'ctrl+o':
            live = [b for b in self.comp.blocks.values() if not b.committed and b.height > 1]
            if live: self.comp.toggle(live[-1])
            return self.paint()
        self.buf.handle(k)
        self.paint()

    def stream(self, blk, chunk):
        """Grow the reply as the model produces it, repainting the block from its whole text.

        Not `Compositor.extend`, for two reasons: each extend appends at least one row, so a
        token stream would print one word per line; and a tool call that starts mid-reply
        makes the reply no longer the last block, which extend refuses. Repainting from the
        accumulated text is correct under both, and a reply is small enough to repaint.
        """
        self._reply = (self._reply + chunk) if blk is not None else chunk
        if blk is None: return self.say(Text(self._reply), 'reply', fold=None)
        self.comp.set_body(blk, Text(self._reply))
        self.comp.refresh_block(blk)
        return blk

In [ ]:
#| export
@patch
def tail(self:Ui):
    "The live-tail description shared with Teleprint's transcript view."
    pre = self.ASKING if self.ask is not None else '▌ '
    rows = [self.status()]
    if self.hint: rows.append(Text(' ' + self.hint, style=GRUVBOX['gray']))
    prompt = self.prompt()
    rows.append(prompt)
    before = Text(pre + self.buf.text[:self.buf.cursor])
    rendered = self.comp.console.render_lines(before, pad=False)
    cursor = (len(rows) - 1, len(rendered) - 1, sum(s.cell_length for s in rendered[-1]))
    return rows, cursor

@patch
def recall(self:Ui, step):
    "Move through submitted prompts, preserving the draft beyond the newest entry."
    if not self.history: return False
    if self.history_at == len(self.history): self.draft = self.buf.text
    self.history_at = max(0, min(len(self.history), self.history_at + step))
    text = self.draft if self.history_at == len(self.history) else self.history[self.history_at]
    self.buf.text, self.buf.cursor = text, len(text)
    return True

_core_submit = Ui.submit

@patch
def submit(self:Ui):
    "Submit the line and remember it for Ctrl-P/Ctrl-N recall."
    line = self.buf.text.strip()
    if line and (not self.history or self.history[-1] != line): self.history.append(line)
    self.history_at, self.draft = len(self.history), ''
    return _core_submit(self)

_core_on_key = Ui.on_key

@patch
def on_key(self:Ui, k):
    "Give transcript navigation priority; keep prompt recall on Ctrl-P/Ctrl-N."
    view = self.transcript
    if view.active:
        if view.on_key(k): return None
        if k.name == 'escape':
            view.leave()
            return self.paint()
        if k.name == 'enter':
            if not view.composing: return None
            out = self.submit()
            view.leave()
            self.paint()
            return out
        out = _core_on_key(self, k)
        view.rebuild(bottom=view.follow)
        return out
    if (k.name == 'ctrl+r' or (k.name in ('up', 'down') and not self.buf.text)) and self.ask is None:
        view.enter()
        if k.name in ('up', 'down'): view.on_key(k)
        return None
    if k.name in ('ctrl+p', 'ctrl+n'):
        self.recall(-1 if k.name == 'ctrl+p' else 1)
        return self.paint()
    return _core_on_key(self, k)

@patch
def paste(self:Ui, text):
    "Insert text, or attach an image when the paste is a readable image path."
    raw = text.strip().strip('[]')
    path = Path(raw)
    if path.suffix.lower() in {'.png', '.jpg', '.jpeg', '.gif', '.webp'} and path.is_file():
        self.image = path.read_bytes()
        self.hint = f'image attached: {path.name}'
    else:
        self.buf.insert(text)
    if self.transcript.active:
        self.transcript.composing = True
        return self.transcript.rebuild(bottom=self.transcript.follow)
    return self.paint()

@patch
def reply(self:Ui, text):
    "A model reply rendered as GitHub Dark Markdown, including syntax-highlighted code."
    return Markdown(text, code_theme='github-dark', style=GRUVBOX['fg1'])

@patch
def stream(self:Ui, blk, chunk):
    "Grow and repaint one Markdown reply as model chunks arrive."
    self._reply = (self._reply + chunk) if blk is not None else chunk
    body = self.reply(self._reply)
    if blk is None: return self.say(body, 'reply', fold=None)
    self.comp.set_body(blk, body)
    self.comp.refresh_block(blk)
    return blk

Assistant Markdown keeps distinct semantic styles instead of flattening everything to terminal foreground. The model's source remains plain Markdown; only its rendering gains color, so raw Rich markup cannot escape into the terminal.

Some requests have more than one honest reading, and the expensive part is finding out
which one you got. `options_for` decides whether a typed line earns a row of choices,
`ChoiceMenu` owns navigating and picking one, and `Ui` decides what the pick *means* -- so
the component is reusable for the next question rather than wired to this one. Each `Option`
carries the instruction it appends, because a preference the model may weigh against the
rest of its briefing is not a choice the user made.

In [ ]:
menu = ChoiceMenu(*options_for('refactor the runtime'))
test_eq(menu.choose('down'), (False, None))          # navigation is not a decision
test_eq(menu.choose('enter')[1].label, 'plan')
test_eq(ChoiceMenu('x', REFACTOR).choose('s')[1].label, 'scope')   # its own letter picks it
test_eq(ChoiceMenu('x', REFACTOR).choose('escape'), (True, None))
test_eq(options_for('what does the runtime do?'), None)            # an ordinary question gets no row
print('\n'.join(r.plain for r in menu.render()))

In [ ]:
from rich.console import Console
color_console = Console(width=72, theme=MARKDOWN_THEME, color_system='truecolor')
styled = Markdown('# Heading\n\nUse `search_code` and **verify** it.\n\n```python\nanswer = 42\n```',
                  code_theme='github-dark')
lines = color_console.render_lines(styled, pad=False)
colors = {seg.style.color.triplet for line in lines for seg in line
          if seg.style and seg.style.color and seg.style.color.triplet}
inline = next(seg for line in lines for seg in line if seg.text == 'search_code')
test_eq(inline.style.bgcolor, None)
test_eq(len(colors) >= 3, True)

A headless terminal, a real compositor, and an agent whose model is a fake: the whole
surface, in a notebook.

In [ ]:
tty = EmuTty(72, 14)
comp = Compositor(tty)
comp._register_signals = lambda: None  # nbdev executes this async cell on a worker thread
await comp.start()
agent, be = fake_agent(replies=['`threshold` is in `ramabana/runtime.py`, and it caps the reserve.'])
ui = Ui(comp, agent)
comp.on_key = ui.on_key          # a coroutine returned by a handler is spawned by the dispatcher
ui.paint()
print(tty.term.text())

 ornith-9b · idle · 14 tools · 15 skills · 0% ctx
›


In [ ]:
ui.buf.insert('refactor the runtime')
test_eq(ui.submit(), None)                    # a row, not a turn
ui.paint()
assert 'a  apply' in tty.term.text() and 'p  plan' in tty.term.text()
test_eq(ui.menu.choose('escape'), (True, None))
ui.menu, ui.menu_prompt = None, ''
ui.buf.clear()
ui.paint()

Cancelling hands the line back to the composer rather than throwing it away: the user typed
it, and the row was the harness's question, not theirs.

In [ ]:
ui.buf.insert('restructure the compactor')
ui.submit()
ui.on_key(Key('escape'))
test_eq(ui.buf.text, 'restructure the compactor')   # given back, still editable
ui.buf.clear()
ui.paint()

That is the resting state -- the status bar says what is loaded and how full it is, and the
cursor sits on the prompt. Everything above the last two rows is the transcript.

In [ ]:
test_eq(tty.term.text().splitlines()[-1], '▌')
[l for l in tty.term.text().splitlines() if 'tools' in l]

[' ornith-9b · idle · 14 tools · 15 skills · 0% ctx']

Typing goes through the real key parser: these are the bytes a terminal sends, not
synthesised `Key` objects.

In [ ]:
comp.on_bytes(b'where is the threshold?')
tty.term.text().splitlines()[-1]

'› where is the threshold?'

The working marker advances independently of model output, so a silent network wait still looks alive. When no turn is running it returns to a stable ready dot.

In [ ]:
resting_status = ui.status().plain
ui.turn = object()
first = ui.status().plain
ui.frame += 1
second = ui.status().plain
ui.turn = None
test_eq('● ' in resting_status and 'WORKING' not in resting_status, True)
test_eq('⠋ working' in first, True)
test_eq('⠙ working' in second, True)

Enter prints the question as a block and spawns the turn. Awaiting a moment lets the model
thread run -- in the real app that wait is the event loop, which is why nothing here blocks
it.

In [ ]:
comp.on_bytes(b'\r')
await asyncio.sleep(0.4)
print(tty.term.text())

› where is the threshold?
· 🔍 Search where is the threshold?
  no matches (memory)
  `threshold` is in `ramabana/runtime.py`, and it caps the reserve.
 ornith-9b · ready · 14 tools · 15 skills · 2% ctx · 15 tok · in 10 ·
out 5 · model
›


In [ ]:
ui.buf.text, ui.buf.cursor = 'x' * 80, 80
ui.paint()
test_eq(comp._cursor[1], len(ui.buf.text) % tty.size[0])
test_eq(0 <= comp._cursor[0] < comp.rows, True)
ui.buf.text, ui.buf.cursor = 'top\nbottom', len('top\nbottom')
ui.paint()
test_eq(comp._cursor[1], len('bottom'))
test_eq(0 <= comp._cursor[0] < comp.rows, True)
ui.buf.clear()
ui.paint()

Prompt history uses Ctrl-P/Ctrl-N, preserving a draft. With an empty composer, Up or Down enters Teleprint's transcript browser instead.

In [ ]:
ui.buf.insert('first prompt')
ui.submit()
ui.buf.insert('second prompt')
ui.submit()
ui.buf.insert('unfinished draft')
ui.on_key(Key('ctrl+p'))
test_eq(ui.buf.text, 'second prompt')
ui.on_key(Key('ctrl+p'))
test_eq(ui.buf.text, 'first prompt')
ui.on_key(Key('ctrl+n'))
ui.on_key(Key('ctrl+n'))
test_eq(ui.buf.text, 'unfinished draft')
ui.buf.clear()
ui.on_key(Key('up'))
test_eq(ui.transcript.active, True)
ui.on_key(Key('g'))
test_eq(ui.transcript.top, 0)
ui.on_key(Key('escape'))
test_eq(ui.transcript.active, False)

The transcript now holds the question, the tool calls the agent made on the way, and the
reply -- each one a block with its own gutter.

In [ ]:
[(b.tag, b.height) for b in comp.blocks.values()]

[('user', 1), ('tool', 2), ('reply', 1)]

In [ ]:
test_eq(agent.calls[0][0], 'search_code')       # the preflight ran, and the feed recorded it
test_eq(any('threshold' in str(b.body) for b in comp.blocks.values()), True)
be.sent and len(be.sent)

1

## Folding

A block taller than `FOLD` is born folded, so a four-hundred-line file view costs one row
until it is asked for. Ctrl-O toggles the last block; mouse clicks remain available for
normal terminal selection.

In [ ]:
long = ui.say(Text('\n'.join(f'line {i}' for i in range(40))), 'tool')
long.collapsed, long.height

(True, 40)

In [ ]:
comp.toggle(long)
test_eq(long.collapsed, False)
comp.toggle(long)
long.collapsed

True

A one-line block has nothing to hide, and toggling it is a no-op rather than an error.

In [ ]:
short = ui.say(Text('one line'), 'tool')
comp.toggle(short), short.collapsed

(None, False)

## Approvals in a terminal

The gate blocks the model's worker thread until a person answers, so the request arrives on
that thread and the answer comes from the loop. The input line becomes the answer: type a
reason, then `y`, `n`, or `a` for the rest of the session.

A write, asked for from another thread -- exactly as a tool call does it:

In [ ]:
tty2 = EmuTty(72, 12)
comp2 = Compositor(tty2)
comp2._register_signals = lambda: None  # headless notebook test; no process signals to own
await comp2.start()
gated, _ = fake_agent(replies=['done'])
gated.approvals = Approvals(tools=WRITE_TOOLS, host=gated.host)
ui2 = Ui(comp2, gated)
comp2.on_key = ui2.on_key
ui2.paint()
threading.Thread(target=lambda: gated.approvals.request(
    'create_file', {'path': '/proj/notes.md', 'text': '# notes\n'}), daemon=True).start()
await asyncio.sleep(0.2)
print(tty2.term.text())

? create_file → /proj/notes.md

  /proj/notes.md  (new file, 8 chars)

  # notes

 ornith-9b · idle · 14 tools · 15 skills · 0% ctx
approve? [y/n/a, or a reason + enter]


The preview is what the person reads: the path, whether it overwrites, and the head of what
would be written. The prompt has become the approval question.

In [ ]:
test_eq(ui2.ask.tool, 'create_file')
tty2.term.text().splitlines()[-1]

'approve? [y/n/a, or a reason + enter]'

`y`, `n` and `a` answer only while nothing has been typed -- a reason like "put it in docs/
instead" contains all three letters, so once there is text the letters are text and `enter`
is the answer. Typing a reason and pressing it refuses *with* that reason, which is the whole
point of the gate: something the model can act on rather than the word "denied".

In [ ]:
comp2.on_bytes(b'put it in docs/ instead')
tty2.term.text().splitlines()[-1]

'approve? [y/n/a, or a reason + enter] put it in docs/ instead'

In [ ]:
comp2.on_bytes(b'\r')
await asyncio.sleep(0.2)
ui2.ask, gated.approvals.history[-1].reply()

(None, 'Denied by human operator. Reason given: put it in docs/ instead')

Prompt history belongs to the composer, while transcript history belongs to Teleprint's block model. Up and Down recall submitted prompts only when `Buffer` cannot move within a multiline prompt; Ctrl-R enters Teleprint's transcript view, where old folded blocks can be searched, expanded, and copied without creating a second history.

In [ ]:
# Helpers are defined beside `Ui`, before the executable surface examples.

In [ ]:
test_eq(bool(gated.approvals.history[-1]), False)
test_eq(tty2.term.text().splitlines()[-1], '▌')      # the prompt is back
[b.tag for b in comp2.blocks.values()][-2:]

['ask', 'note']

`a` is the deliberate bulk answer: it approves this request and switches the policy to
`auto` for the rest of the session, which is a thing a person does on purpose rather than by
holding down return.

In [ ]:
threading.Thread(target=lambda: gated.approvals.request('create_file', {'path': '/proj/b.md'}), daemon=True).start()
await asyncio.sleep(0.2)
comp2.on_bytes(b'a')
await asyncio.sleep(0.2)
gated.approvals.mode, gated.approvals.history[-1].note

('auto', 'approved for the rest of this session')

## Commands

Slash commands are answered by the agent, not by the terminal: there is one implementation of
`/model`, and it is not in a frontend. `/help` is the only one this layer owns, because the
keys it describes are this layer's.

In [ ]:
ui.buf.insert('/tools')
ui.submit()
[l for l in str(comp.blocks[max(comp.blocks)].body[0]).splitlines()[:4]]

['create_file', 'create_skill', 'delegate_parallel', 'delegate_search']

In [ ]:
ui.buf.insert('/nope')
test_eq(ui.submit(), None)
comp.blocks[max(comp.blocks)].tag

'error'

A command is recognised before the options row, so `/tools refactor` switches nothing on and
runs the command -- otherwise any command with the word in it would open a menu instead.

In [ ]:
ui.buf.insert('/tools refactor')
test_eq(ui.submit(), None)
test_eq(ui.menu, None)                    # a command, never a question
comp.blocks[max(comp.blocks)].tag

An empty line does nothing at all, rather than sending an empty turn to the model.

In [ ]:
test_eq(ui.submit(), None)
ui.buf.text

''

## Running it

`mk_agent` is the assembly: a `LocalHost` over the folders named on the command line, an
`Agent` over that, and the approval gate wired to both. `amain` is the tty loop, and it is
short because everything it could get wrong lives in `Ui`.

In [ ]:
#| export
def mk_host(roots=('.',),
            approvals=None,          # an `Approvals` for the host to put writes in front of
            web=True,                # wire the web tools to fossick
            vault=False,             # keep what is read in a vishalakshi vault, for the next session
            read_outside=False):     # let the read-only tools name any path on this machine
    """The host both frontends run on: `LocalHost`, or `VaultHost` when memory should outlive it.

    `vishalakshi` is imported only when it is asked for, so a terminal session that does not
    want durable memory does not pay for an embedding model to find that out.
    """
    if vault:
        from .vault import VaultHost as Host
    else: Host = LocalHost
    return Host(roots, approvals=approvals, web=web, read_outside=read_outside)


def mk_agent(roots=('.',),
             model=None,
             approve='ask',           # ask | auto | off | none (gate nothing at all)
             web=True,                # wire the web tools to fossick
             vault=False,             # keep what is read in a vishalakshi vault, for the next session
             read_outside=False,      # let the read-only tools name any path on this machine
             **kw):                   # forwarded to `Agent`
    "A host over the named folders and an `Agent` over that, gated the way `approve` says."
    approvals = None if approve == 'none' else Approvals(tools=WRITE_TOOLS, mode=approve)
    host = mk_host(roots, approvals=approvals, web=web, vault=vault, read_outside=read_outside)
    # The gate previews `create_file` by asking the host whether the path already exists, so
    # "new file" and "OVERWRITES an existing file" are different sentences to approve.
    if approvals is not None: approvals.host = host
    return Agent(host, model=model, approvals=approvals, **kw), host

In [ ]:
#| export
async def amain(agent, hint=''):
    "The tty loop: one terminal, one event loop, one place that owns the keyboard."
    tty = RealTty()
    tty.write('\x1b[?1000;1006h\x1b[?2004h')  # SGR mouse + bracketed paste
    done = asyncio.Event()
    try:
        comp = await Compositor(tty).start()
        ui = Ui(comp, agent, loop=asyncio.get_running_loop())
        ui.hint = hint
        comp.on_task_error = lambda e, t: ui.say(Text(f'{t.get_name()} failed: {e!r}'), 'error')
        comp.spawn(ui.animate(), name='spinner')
        def on_key(k):
            out = ui.on_key(k)
            if out == 'quit': return done.set()
            if out is not None:
                ui.turn = comp.spawn(out, name='turn')
                ui.paint()
        comp.on_key = on_key
        comp.on_paste = ui.paste
        comp.on_resize = lambda: (comp.resize(), ui.paint())
        comp.on_mouse = ui.transcript.on_mouse
        brand = Text('RAMABANA', style=f"bold {GRUVBOX['fg0']}")
        brand.append(f"  {agent.note}", style=GRUVBOX['gray'])
        ui.say(brand + Text(f'\n\n{HELP}', style=GRUVBOX['gray']), 'note', fold=None)
        ui.paint()
        loop = asyncio.get_running_loop()
        loop.add_reader(tty.fd, lambda: comp.on_bytes(tty.read(timeout=0)))
        try:
            while not done.is_set():          # the key parser needs periodic flushes (esc disambiguation)
                try: await asyncio.wait_for(done.wait(), 0.2)
                except asyncio.TimeoutError: comp.flush_input()
        finally:
            loop.remove_reader(tty.fd)
            comp.stop()
    finally:
        tty.write('\x1b[?2004l\x1b[?1000;1006l\r\n')
        tty.restore()
        agent.close()

The agent that reaches the terminal is a real one over real folders, gated on writes:

In [ ]:
tempdir = tempfile.mkdtemp()
cli_agent, cli_host = mk_agent([tempdir], approve='ask', web=False)
cli_host.roots, sorted(WRITE_TOOLS & set(t.__name__ for t in cli_agent.tools))

(['/private/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmpr4g2iv5i'],
 ['add_cell',
  'create_file',
  'create_skill',
  'edit_cell',
  'edit_file',
  'run_python'])

In [ ]:
test_eq(cli_agent.approvals.mode, 'ask')
test_eq(mk_agent([tempdir], approve='none')[0].approvals, None)
cli_agent.model.name

'ornith-9b'

One turn with no terminal at all, for a pipe or a script -- and the entry point itself, which
is `ramabana` on the command line.

In [ ]:
#| export
def ask_once(agent, prompt):
    "One turn with no terminal at all, for a pipe or a script. Returns the exit code."
    print(agent.ask(prompt))
    ok, problems = agent.ready, list(agent.problems)
    for p in problems: print(f'! {p}', file=sys.stderr)
    agent.close()
    return 0 if ok else 1

In [ ]:
#| export
@call_parse
def main(
    prompt: str = '',                    # run one turn and exit; omit for the interactive session
    root: str = '.',                     # folders the agent may touch, comma separated
    model: str = None,                   # the turn model; the routing default when omitted
    approve: str = 'ask',                # ask | auto | off | none (gate nothing at all)
    web: bool = True,                    # let the web tools reach the network through fossick
    read_outside: bool = False,          # let reads name any path on this machine; writes stay inside
    vault: bool = False,                 # keep what is read in a vishalakshi vault, for the next session
    cfg: str = '~/.config/ramabana',     # config dir, for skills, extensions and resumable history
    resume: str = '',                    # saved session id/prefix, or 'latest'
):
    "Ramabana in a terminal: a coding agent over the folders you name."
    roots = [r.strip() for r in str(root).split(',') if r.strip()]
    agent, host = mk_agent(roots, model=model, approve=approve, web=web, vault=vault,
                           read_outside=read_outside,
                           cfg=Path(cfg).expanduser() if cfg else None)
    if resume:
        try: agent.resume_session(resume)
        except Exception as e:
            print(f'could not resume: {agent_err(e)}', file=sys.stderr)
            return 2
    if agent.start() is None and not prompt:
        print(f'no model available: {agent.note}', file=sys.stderr)
    if prompt: return sys.exit(ask_once(agent, prompt))
    hint = f"{', '.join(host.roots)} · /help"
    try: asyncio.run(amain(agent, hint))
    except KeyboardInterrupt: pass

`ask_once` prints the answer and reports anything that went wrong on stderr, so
`ramabana -p 'what changed?' | less` behaves like a Unix program.

In [ ]:
piped, _ = fake_agent(replies=['nothing changed since the last commit.'])
test_eq(ask_once(piped, 'what changed?'), 0)
piped.note

nothing changed since the last commit.


'fake · local · 1k ctx · 14 tools'

The command line, as `--help` prints it:

In [ ]:
#| eval: false
!ramabana --help

usage: ramabana [-h] [--prompt PROMPT] [--root ROOT] [--model MODEL]
                [--approve APPROVE] [--no-web] [--cfg CFG]

Ramabana in a terminal: a coding agent over the folders you name.

options:
  -h, --help         show this help message and exit
  --prompt PROMPT    run one turn and exit; omit for the interactive session
                     (default: )
  --root ROOT        folders the agent may touch, comma separated (default: .)
  --model MODEL      the turn model; the routing default when omitted
  --approve APPROVE  ask | auto | off | none (gate nothing at all) (default:
                     ask)
  --no-web           let the web tools reach the network through fossick
                     (default: True)
  --cfg CFG          config dir, for skills, extensions and history


```
usage: ramabana [-h] [--prompt PROMPT] [--root ROOT] [--model MODEL]
                [--approve APPROVE] [--no-web] [--read_outside] [--vault]
                [--cfg CFG] [--resume RESUME]

Ramabana in a terminal: a coding agent over the folders you name.

  --prompt         run one turn and exit; omit for the interactive session
  --root           folders the agent may touch, comma separated (default: .)
  --model          the turn model; the routing default when omitted
  --approve        ask | auto | off | none (gate nothing at all) (default: ask)
  --no-web         keep the web tools off the network
  --read_outside   let reads name any path on this machine; writes stay inside
  --vault          keep what is read in a vishalakshi vault, for the next session
  --cfg            config dir, for skills, extensions and history
  --resume         saved session id/prefix, or 'latest'
```

So a session over this repository, on a local model, with nothing gated:

```bash
ramabana --root . --model qwen-4b --approve none
```

One that may read the rest of the machine but only write here:

```bash
ramabana --root . --read_outside
```

And one turn for a pipe, with no terminal at all:

```bash
ramabana --prompt 'which files import fastllm?'
```

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()